<a href="https://colab.research.google.com/github/ekBlaise/Non_instruction_Causal_LM_Fine_tuning/blob/master/Non_instruction_Causal_LM_Fine_tuning_or_Domain_Adaptive_Continued_Pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Pharma PDF → Raw Text → Non-Instruction Causal LLM Fine-Tuning

Non-instruction Causal LM Fine-tuning or Domain-Adaptive Continued Pretraining

Causal meaning: Past causes future prediction.

#### Goal: Use a pharma-domain PDF as raw text data and perform **non-instruction causal language model fine-tuning** using LoRA/QLoRA. **bold text**

## Pipeline

```text
Pharma PDF
   ↓
PDF text extraction
   ↓
Text cleaning and normalization
   ↓
Data creation
   ↓
Hugging Face Dataset Conversion
   ↓
Tokenization
   ↓
LoRA/QLoRA fine-tuning
   ↓
Validation loss
   ↓
Adapter saving and reloading
   ↓
Text continuation inference
```

## Continued Pretraining vs Instruction Fine-Tuning

In this notebook, we are performing **continued pretraining / non-instruction fine-tuning** on raw pharma PDF text.

The model is given raw domain text such as:

> Metformin is one of the most widely prescribed oral antihyperglycemic agents...

The model then learns to **predict the next token** from this raw text.

This means the model learns:

- Pharma language
- Drug names
- Medical terminology
- Scientific writing style
- Domain-specific sentence patterns

However, the model is **not explicitly taught**:

- How to answer a user's question
- How to follow instructions
- How to respond in Q&A format
- How to behave like a domain-specific chatbot

---

## What Instruction Fine-Tuning Looks Like

In instruction fine-tuning, the training data is prepared in an **instruction-response format**.

Example:

```json
{
  "instruction": "Explain the mechanism of action of Metformin.",
  "input": "",
  "output": "Metformin primarily activates AMPK, which improves glucose uptake and reduces hepatic gluconeogenesis."
}

{
  "messages": [
    {
      "role": "user",
      "content": "What is the primary mechanism of action of Metformin?"
    },
    {
      "role": "assistant",
      "content": "Metformin primarily works by activating AMPK..."
    }
  ]
}

## Pipeline

```text
Non-instrcution FT(RAW)
      ↓
will save the model
      ↓
load the model
      ↓
I will perform instruction FT on same model(question/answer data)
      ↓
will save our model
      ↓
again will load the same model
      ↓
will perform the preference tuning on top of it(choosed/ reject data)
   
```

we are going to train the LORA adapter

In [ ]:
# ============================================================
# 1. Install required libraries
# ============================================================
# PyMuPDF: PDF text extraction
# datasets: Hugging Face dataset creation
# transformers/accelerate: model, tokenizer, Trainer
# peft: LoRA/QLoRA adapters
# bitsandbytes: 4-bit/8-bit quantized loading

!pip install -q -U pymupdf datasets transformers accelerate peft bitsandbytes sentencepiece

In [ ]:
# ============================================================
# 2. Imports
# ============================================================

import os
import re
import gc
import math
import json
import random
import unicodedata
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

import fitz  # PyMuPDF
import torch
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)

In [ ]:
# ============================================================
# 3. Global configuration
# ============================================================
# Keep all important parameters in one place.
# This makes the notebook easier to debug, reproduce, and productionize.

@dataclass
class Config:
    # Upload your pharma PDF to this path in Colab or update this value.
    pdf_path: str = "/content/Metformin-Lipid-Therapy-Knowledge.pdf"

    # TinyLlama is selected because it is lightweight for classroom use.
    model_name: str = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

    # Output directories.
    output_dir: str = "/content/pharma_tinyllama_lora_output"
    adapter_dir: str = "/content/pharma_tinyllama_lora_adapter"
    processed_data_dir: str = "/content/pharma_processed_data"

    # Text preprocessing.
    min_chars_per_paragraph: int = 80
    block_size: int = 512

    # Train/eval split.
    test_size: float = 0.15
    seed: int = 42

    # LoRA parameters.
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05

    # Training parameters.
    num_train_epochs: float = 3.0
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    logging_steps: int = 5
    eval_steps: int = 10
    save_steps: int = 25
    save_total_limit: int = 2
    # For a quick demo, set max_steps to 20 or 30.
    # For a complete run, keep it as -1.
    max_steps: int = -1

In [ ]:
config = Config()

In [ ]:
config

In [ ]:
print(json.dumps(asdict(config), indent=2))

In [ ]:
config.output_dir

In [ ]:
config.processed_data_dir

In [ ]:
os.makedirs(config.output_dir, exist_ok=True)
os.makedirs(config.adapter_dir, exist_ok=True)
os.makedirs(config.processed_data_dir, exist_ok=True)

In [ ]:
# ============================================================
# 4. Optional Colab upload helper
# ============================================================
# Run this cell only if your PDF is not already present at config.pdf_path.

if not os.path.exists(config.pdf_path):
    print(f"PDF not found at: {config.pdf_path}")
else:
    print(f"PDF found: {config.pdf_path}")

In [ ]:
# # ============================================================
# # 5. Extract text from PDF
# # ============================================================
def extract_pdf_pages(pdf_path: str) -> List[Dict[str, Any]]:
    # Extract page-level text from a PDF.
    pages = []
    with fitz.open(pdf_path) as doc:
        for page_index, page in enumerate(doc, start=1):
            text = page.get_text("text")
            text = text.strip() if text else ""
            if text:
                pages.append({
                    "page": page_index,
                    "text": text,
                    "char_count": len(text),
                })
    return pages

In [ ]:
config.pdf_path

In [ ]:
pdf_pages = extract_pdf_pages(config.pdf_path)

In [ ]:
print(f"Total pages with extracted text: {len(pdf_pages)}")
print("Page-level character counts:")
for item in pdf_pages:
    print(f"Page {item['page']}: {item['char_count']} characters")

In [ ]:
print(pdf_pages[0]["text"])

| Cleaning Step                          | Code / Logic                             | What It Does                                                                  | Example Before                                                  | Example After                                                  | Why It Matters for Fine-Tuning                                            |
| -------------------------------------- | ---------------------------------------- | ----------------------------------------------------------------------------- | --------------------------------------------------------------- | -------------------------------------------------------------- | ------------------------------------------------------------------------- |
| Unicode normalization                  | `unicodedata.normalize("NFKC", text)`    | Converts unusual Unicode characters into standard readable characters.        | `ＡＭＰＫ`, `ﬁ`                                                     | `AMPK`, `fi`                                                   | Prevents tokenizer confusion caused by hidden or non-standard characters. |
| Remove zero-width characters           | `text.replace("\u200b", "")`             | Removes invisible zero-width spaces from PDF text.                            | `Metformin​ activates AMPK`                                     | `Metformin activates AMPK`                                     | Invisible characters can create bad tokens and noisy training data.       |
| Remove BOM / hidden marker             | `text.replace("\ufeff", "")`             | Removes hidden Byte Order Mark characters sometimes found in extracted text.  | `﻿Metformin is used...`                                         | `Metformin is used...`                                         | Keeps the training text clean and consistent.                             |
| Fix hyphenated line breaks             | `re.sub(r"(\w)-\n(\w)", r"\1\2", text)`  | Joins words that were broken across PDF lines.                                | `gluconeogene-\nsis`                                            | `gluconeogenesis`                                              | Prevents the model from learning broken medical terms.                    |
| Normalize spaces and tabs              | `re.sub(r"[ \t]+", " ", text)`           | Converts multiple spaces or tabs into one space.                              | `Metformin     activates    AMPK`                               | `Metformin activates AMPK`                                     | Makes text consistent and easier for tokenizer/model to learn.            |
| Normalize blank lines                  | `re.sub(r"\n{3,}", "\n\n", text)`        | Converts too many blank lines into a proper paragraph gap.                    | `Para 1\n\n\n\nPara 2`                                          | `Para 1\n\nPara 2`                                             | Preserves paragraph structure without unnecessary whitespace noise.       |
| Remove standalone page numbers         | `re.sub(r"(?m)^\s*\d+\s*$", "", text)`   | Removes lines that contain only page numbers.                                 | `1` or `23`                                                     | Removed                                                        | Prevents the model from learning irrelevant PDF page numbers.             |
| Split into paragraphs                  | `re.split(r"\n\s*\n", text)`             | Splits text wherever there is a blank line.                                   | `Para 1\n\nPara 2`                                              | `["Para 1", "Para 2"]`                                         | Helps preserve meaningful document structure.                             |
| Remove line wrapping inside paragraphs | `re.sub(r"\n+", " ", paragraph)`         | Converts broken lines inside the same paragraph into a single paragraph line. | `Metformin is widely prescribed\noral antihyperglycemic agent.` | `Metformin is widely prescribed oral antihyperglycemic agent.` | Prevents the model from learning artificial PDF line breaks.              |
| Normalize paragraph spacing            | `re.sub(r"\s+", " ", paragraph).strip()` | Removes extra spaces inside each paragraph and trims start/end spaces.        | `  Metformin   activates   AMPK.  `                             | `Metformin activates AMPK.`                                    | Produces clean, readable training examples.                               |
| Remove empty paragraphs                | `if paragraph:`                          | Keeps only non-empty cleaned paragraphs.                                      | `""`                                                            | Removed                                                        | Avoids useless blank samples in the dataset.                              |
| Rebuild cleaned text                   | `"\n\n".join(cleaned_paragraphs)`        | Joins cleaned paragraphs with two newlines.                                   | List of cleaned paragraphs                                      | Clean paragraph-level text                                     | Creates a clean corpus suitable for causal LM training.                   |
| Track cleaned page length              | `char_count: len(cleaned_text)`          | Stores number of characters after cleaning.                                   | Raw page length unknown                                         | `char_count = 1450`                                            | Helps debug whether a page has too little or too much extracted content.  |
| Preview cleaned output                 | `cleaned_pages[0]["text"][:1500]`        | Prints first 1500 characters of cleaned page 1.                               | Full cleaned page                                               | Preview text                                                   | Helps manually verify that cleaning worked correctly.                     |


In [ ]:
# ============================================================
# 6. Text cleaning utilities
# ============================================================

In [ ]:
# Ye hidden/weird Unicode characters ko normal form me convert karta hai.

# Example:

# ＡＭＰＫ → AMPK
# ﬁ → fi
def normalize_unicode(text: str) -> str:
    # Normalize Unicode characters to reduce hidden character issues.
    return unicodedata.normalize("NFKC", text)

In [ ]:
def clean_pdf_text(text: str) -> str:
    # Clean raw PDF text while preserving meaningful scientific content.
    text = normalize_unicode(text)

    # Remove zero-width and non-printing characters often found in PDFs.
    # Ye special hidden characters remove karta hai.
    # \u200b
    # Zero-width space hota hai. Ye dikhta nahi hai but text ke andar hota hai.
    # \ufeff
    # Byte Order Mark / hidden marker hota hai. PDF/text extraction me kabhi aa jata hai.
    # Inhe remove karna important hai, warna tokenizer confuse ho sakta hai.
    text = text.replace("\u200b", "")
    text = text.replace("\ufeff", "")

    # Fix hyphenated line breaks: for example, "gluconeogene-\nsis" -> "gluconeogenesis".
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)

    # Convert multiple spaces/tabs into a single space.
    text = re.sub(r"[ \t]+", " ", text)

    # Normalize repeated blank lines.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Remove standalone page numbers.
    text = re.sub(r"(?m)^\s*\d+\s*$", "", text)

    # Preserve paragraph boundaries, but remove line wrapping inside paragraphs.
    paragraphs = re.split(r"\n\s*\n", text)

    cleaned_paragraphs = []

    for paragraph in paragraphs:
        paragraph = re.sub(r"\n+", " ", paragraph)
        paragraph = re.sub(r"\s+", " ", paragraph).strip()
        if paragraph:
            cleaned_paragraphs.append(paragraph)

    return "\n\n".join(cleaned_paragraphs)

In [ ]:
pdf_pages

In [ ]:
cleaned_pages = []

In [ ]:
for page in pdf_pages:
    cleaned_text = clean_pdf_text(page["text"])
    cleaned_pages.append({
        "page": page["page"],
        "text": cleaned_text,
        "char_count": len(cleaned_text),
    })

In [ ]:
cleaned_pages

In [ ]:
print("Cleaned page preview:\n")
print(cleaned_pages[0]["text"])

In [ ]:
cleaned_pages

In [ ]:
len("Split cleaned pages into paragraphs")

In [ ]:
# ============================================================
# 7. Split cleaned pages into paragraphs
# ============================================================
# Paragraph records are clean units for audit and inspection.
# Later, tokenized text will be packed into fixed-size training blocks.

def split_into_paragraph_records(cleaned_pages: List[Dict[str, Any]], min_chars: int = 80) -> List[Dict[str, Any]]:
    records = []
    seen = set()

    for page in cleaned_pages:
        paragraphs = re.split(r"\n\s*\n", page["text"])
        for paragraph_index, paragraph in enumerate(paragraphs, start=1):
            paragraph = paragraph.strip()
            if len(paragraph) < min_chars:
                continue

            # Deduplicate exact repeated paragraphs.
            normalized_key = re.sub(r"\s+", " ", paragraph.lower()).strip()
            if normalized_key in seen:
                continue
            seen.add(normalized_key)

            records.append({
                "text": paragraph,
                "source_page": page["page"],
                "paragraph_id": paragraph_index,
                "char_count": len(paragraph),
            })

    return records

In [ ]:
config.min_chars_per_paragraph

In [ ]:
paragraph_records = split_into_paragraph_records(cleaned_pages, min_chars=config.min_chars_per_paragraph)

In [ ]:
print(f"Total paragraph records: {len(paragraph_records)}")
for i, record in enumerate(paragraph_records[:5]):
    print("=" * 100)
    print(f"Record {i} | Page {record['source_page']} | Characters: {record['char_count']}")
    print(record["text"])

In [ ]:
paragraph_records

In [ ]:
config.processed_data_dir

In [ ]:
# ============================================================
# 8. Save extracted and cleaned corpus for auditability
# ============================================================
# In real projects, always save intermediate datasets.
# This helps with reproducibility, debugging, and compliance review.

raw_pages_path = os.path.join(config.processed_data_dir, "pdf_pages_raw.jsonl")
paragraphs_path = os.path.join(config.processed_data_dir, "pharma_paragraph_process.jsonl")

with open(raw_pages_path, "w", encoding="utf-8") as f:
    for item in pdf_pages:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

with open(paragraphs_path, "w", encoding="utf-8") as f:
    for item in paragraph_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"Saved raw pages to: {raw_pages_path}")
print(f"Saved cleaned paragraph corpus to: {paragraphs_path}")

In [ ]:
# ============================================================
# 9. Create Hugging Face Dataset
# ============================================================

if len(paragraph_records) < 2:
    raise ValueError(
        "The extracted corpus is too small. Please provide a larger pharma PDF or lower min_chars_per_paragraph."
    )

text_dataset = Dataset.from_list(paragraph_records)


In [ ]:
print(text_dataset)

In [ ]:
print(text_dataset[0])

In [ ]:
# ============================================================
# 10. Train/eval split
# ============================================================
# Even for small demos, keep an evaluation set.
# This gives us validation loss and perplexity.

split_dataset = text_dataset.train_test_split(test_size=config.test_size, seed=config.seed)

raw_datasets = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print(raw_datasets)

## 11. Load tokenizer

The tokenizer converts text into token IDs.

For causal language modeling, the model learns:

```text
Given previous tokens, predict the next token.
```

This is why we call it **non-instruction causal LM fine-tuning**.

## What Does `512` Mean in Text Packing?

In this notebook, `512` means the **sequence length** or **block size** used for causal language model training.

It is **not the embedding size**.

It simply means:

> Each training example will contain 512 tokens.

---

## Example

Suppose the tokenizer converts our pharma text into 1,300 tokens:

```text
[token_1, token_2, token_3, ..., token_1300]?

if we set:

block_size = 512

then the tokens are split like this:

Block 1 = token 1 to token 512
Block 2 = token 513 to token 1024
Remaining tokens = token 1025 to token 1300

Is 512 Padding?

Not exactly.

512 is the target length of each training block.

If we use text packing, we try to fill each block with real tokens, so padding is reduced.

Without packing:

Paragraph 1 = 100 tokens + 412 padding tokens
Paragraph 2 = 200 tokens + 312 padding tokens

With packing:

Block 1 = 512 real tokens
Block 2 = 512 real tokens

So 512 is the fixed token length used to make training efficient.

Is 512 Embedding Size?

No.

Embedding size means the hidden vector dimension of the model.

For example, a model may convert each token into a vector like:

token → 2048-dimensional vector

That 2048 is embedding/hidden size.

But 512 here means:

How many tokens we give to the model at one time

In [ ]:
config.model_name

In [ ]:
# ============================================================
# 11. Load tokenizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

In [ ]:
# Some Llama-style models do not define a pad token.
# For causal LM fine-tuning, using EOS as PAD is a common practical choice.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer loaded: {config.model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token} | Pad token id: {tokenizer.pad_token_id}")
print(f"EOS token: {tokenizer.eos_token} | EOS token id: {tokenizer.eos_token_id}")

In [ ]:
# ============================================================
# 12. Tokenization and text packing
# ============================================================

def tokenize_function(examples):
    # Tokenize text without padding. Padding is handled dynamically by the collator.
    return tokenizer(examples["text"])

In [ ]:
def group_texts(examples):
    # Concatenate tokenized texts and split into fixed-length blocks.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples["input_ids"])

    # Drop the small remainder so all blocks have exactly block_size tokens.
    total_length = (total_length // config.block_size) * config.block_size

    if total_length == 0:
        return {k: [] for k in concatenated_examples.keys()}

    result = {
        k: [t[i : i + config.block_size] for i in range(0, total_length, config.block_size)]
        for k, t in concatenated_examples.items()
    }

    # For causal LM, labels are the same as input_ids.
    # The model internally shifts labels for next-token prediction.
    result["labels"] = result["input_ids"].copy()
    return result

In [ ]:
tokenized_datasets = raw_datasets.map(
    tokenize_function,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing text corpus",
)

In [ ]:
tokenized_datasets

In [ ]:
final_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    desc=f"Packing tokenized text into blocks of {config.block_size}",
)

In [ ]:
final_datasets

In [ ]:
if len(final_datasets["train"]) == 0:
    raise ValueError(
        "No training blocks were created. Reduce config.block_size or provide a larger PDF corpus."
    )

In [ ]:
# Inspect one packed example.
# This verifies that tokenization and packing are working.
sample = final_datasets["train"][0]

In [ ]:
sample

In [ ]:
print("Keys:", sample.keys())
print("input_ids length:", len(sample["input_ids"]))
print("labels length:", len(sample["labels"]))
print("Decoded sample preview:\n")
print(tokenizer.decode(sample["input_ids"][:250]))

## 13. Load model with QLoRA-friendly configuration

We use 4-bit quantized loading when CUDA is available.

Why?

- Lower GPU memory usage
- Faster experimentation
- Practical for Colab-style environments
- Common industry approach for parameter-efficient fine-tuning

If CUDA is not available, the notebook falls back to normal loading, but training will be slow on CPU.

In [ ]:
# ============================================================
# 13. Load base model
# ============================================================

use_cuda = torch.cuda.is_available()
print("CUDA available:", use_cuda)
if use_cuda:
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Clear memory before loading the model.
gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

if use_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    # Required for stable training with k-bit quantized models.
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

# Disable cache during training to avoid warnings and reduce memory usage.
base_model.config.use_cache = False

print("Base model loaded successfully.")

In [ ]:
# ============================================================
# 14. Apply LoRA adapters
# ============================================================
# LoRA trains a small number of adapter parameters instead of updating all base model weights.
# This is cheaper than full fine-tuning and is widely used in real projects.

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=config.lora_r,
    lora_alpha=config.lora_alpha,
    lora_dropout=config.lora_dropout,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


In [ ]:
model = get_peft_model(base_model, lora_config)

In [ ]:
model.print_trainable_parameters()

## Why Do We Need `DataCollatorForLanguageModeling`?

After tokenization and text packing, our dataset contains token IDs in a training-ready structure.

However, the `Trainer` still needs a component that can take multiple examples from the dataset and convert them into a proper batch during training.

That component is called a **data collator**.

```python
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)
What Does the Data Collator Do?

The data collator prepares mini-batches for the model.

It handles things like:

Collecting multiple training examples together
Padding sequences if required
Converting examples into tensors
Preparing labels for language modeling
Example

Suppose our packed dataset has training examples like this:

Example 1 = 512 tokens
Example 2 = 512 tokens
Example 3 = 512 tokens

During training, the Trainer may take two examples at a time:

Batch = Example 1 + Example 2

The data collator converts them into tensors like:

input_ids shape      = [2, 512]
attention_mask shape = [2, 512]
labels shape         = [2, 512]

This is the format the model expects during training.

Why mlm=False?

mlm means Masked Language Modeling.

Masked Language Modeling is used for BERT-style models.

Example:

Metformin is used for [MASK].

The model predicts the masked word:

diabetes

But we are using TinyLlama, which is a causal language model.

Causal language models learn by predicting the next token from left to right.

Example:

Metformin → is
Metformin is → used
Metformin is used → for
Metformin is used for → diabetes

So we set:

mlm=False

This tells Hugging Face:

Do not use BERT-style masked language modeling. Use causal language modeling instead.

Why Is This Needed Even After Tokenization and Packing?

Tokenization converts text into token IDs.

Text packing groups token IDs into fixed-size blocks.

But the data collator prepares those blocks into actual training batches.

So the flow is:

Raw pharma text
   ↓
Tokenization
   ↓
Token IDs
   ↓
Text packing
   ↓
Fixed-size training blocks
   ↓
Data collator
   ↓
Mini-batches for Trainer
   ↓
Model training

In [ ]:
# ============================================================
# 15. Data collator
# ============================================================

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [ ]:
# ============================================================
# 16. Training arguments
# ============================================================
# These settings are designed for a small classroom/demo run.
# For larger corpora, increase dataset size, epochs, and evaluation frequency carefully.

import inspect
from transformers import TrainingArguments

eval_mode = "steps" if len(final_datasets["validation"]) > 0 else "no"

training_kwargs = dict(
    output_dir=config.output_dir,
    num_train_epochs=config.num_train_epochs,
    max_steps=config.max_steps,
    per_device_train_batch_size=config.per_device_train_batch_size,
    per_device_eval_batch_size=config.per_device_eval_batch_size,
    gradient_accumulation_steps=config.gradient_accumulation_steps,
    learning_rate=config.learning_rate,

    # Use warmup_steps instead of deprecated warmup_ratio.
    # For small demo training, 5-10 warmup steps are enough.
    warmup_steps=5,

    weight_decay=config.weight_decay,
    logging_steps=config.logging_steps,
    eval_steps=config.eval_steps,
    save_steps=config.save_steps,
    save_total_limit=config.save_total_limit,
    fp16=use_cuda,
    bf16=False,
    report_to="none",
    remove_unused_columns=False,
)

# Transformers versions differ:
# Some versions use evaluation_strategy.
# Newer versions may use eval_strategy.
training_args_params = inspect.signature(TrainingArguments.__init__).parameters

if "eval_strategy" in training_args_params:
    training_kwargs["eval_strategy"] = eval_mode
elif "evaluation_strategy" in training_args_params:
    training_kwargs["evaluation_strategy"] = eval_mode

# Keep only arguments supported by the installed Transformers version.
safe_training_kwargs = {
    key: value
    for key, value in training_kwargs.items()
    if key in training_args_params
}

In [ ]:
safe_training_kwargs

In [ ]:
removed_args = set(training_kwargs.keys()) - set(safe_training_kwargs.keys())
if removed_args:
    print("Removed unsupported TrainingArguments:", removed_args)

In [ ]:
training_args = TrainingArguments(**safe_training_kwargs)

In [ ]:
print(training_args)

In [ ]:
# ============================================================
# 17. Build Trainer
# ============================================================
# Different Transformers versions support different Trainer arguments.
# Some versions support `tokenizer`, newer versions may use `processing_class`,
# and some lightweight/modified versions support neither.
# So we build Trainer arguments safely based on the installed version.

import inspect
from transformers import Trainer

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"] if len(final_datasets["validation"]) > 0 else None,
    data_collator=data_collator,
)

trainer_params = inspect.signature(Trainer.__init__).parameters

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

safe_trainer_kwargs = {
    key: value
    for key, value in trainer_kwargs.items()
    if key in trainer_params
}

removed_trainer_args = set(trainer_kwargs.keys()) - set(safe_trainer_kwargs.keys())
if removed_trainer_args:
    print("Removed unsupported Trainer arguments:", removed_trainer_args)

In [ ]:
trainer = Trainer(**safe_trainer_kwargs)

print("Trainer is ready.")

In [ ]:
# ============================================================
# 18. Start training
# ============================================================

train_result = trainer.train()

print("Training completed.")
print(train_result)

In [ ]:
# ============================================================
# 19. Save adapter and tokenizer
# ============================================================

trainer.model.save_pretrained(config.adapter_dir)
tokenizer.save_pretrained(config.adapter_dir)

print(f"LoRA adapter saved to: {config.adapter_dir}")
print("Saved files:")
print(os.listdir(config.adapter_dir))

In [ ]:
# ============================================================
# 20. Push LoRA adapter to Hugging Face Hub
# ============================================================

repo_id = "YOUR_HF_USERNAME/pharma-tinyllama-domain-lora"

model.push_to_hub(
    repo_id,
    private=True
)

tokenizer.push_to_hub(
    repo_id,
    private=True
)

print(f"LoRA adapter pushed to: https://huggingface.co/{repo_id}")

In [ ]:
# ============================================================
# 21. Reload base model + LoRA adapter correctly
# ============================================================

# Clean old objects to free memory.
del trainer
try:
    del model
    del base_model
except NameError:
    pass

gc.collect()
if use_cuda:
    torch.cuda.empty_cache()

In [ ]:
if use_cuda:
    reload_bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        quantization_config=reload_bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    inference_base_model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=torch.float32,
        trust_remote_code=True,
    )

inference_tokenizer = AutoTokenizer.from_pretrained(config.adapter_dir, use_fast=True)
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

inference_model = PeftModel.from_pretrained(inference_base_model, config.adapter_dir)
inference_model.eval()

print("Base model + LoRA adapter loaded successfully for inference.")

In [ ]:
# ============================================================
# 22. Inference helper
# ============================================================
# Since this is non-instruction fine-tuning, prompts should look like text continuations,
# not chat-style questions.

def generate_completion(prompt: str, max_new_tokens: int = 120) -> str:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = inference_tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=inference_tokenizer.eos_token_id,
            eos_token_id=inference_tokenizer.eos_token_id,
        )

    return inference_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
# ============================================================
# 23. Test text continuation
# ============================================================
# These prompts are continuation-style prompts.
# In Notebook 2, we will create instruction prompts for Q&A.

prompts = [
    "Metformin is one of the most widely prescribed oral antihyperglycemic agents",
    "Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe",
    "Artificial intelligence is transforming pharmaceutical research by accelerating",
]

for prompt in prompts:
    print("=" * 100)
    print("PROMPT:")
    print(prompt)
    print("\nMODEL CONTINUATION:")
    print(generate_completion(prompt, max_new_tokens=120))
    print()

In [ ]:
# ============================================================
# 24. Optional merge step
# ============================================================
# Uncomment only if you need a merged model for deployment.

merged_model_dir = "/content/pharma_tinyllama_merged_model"
merged_model = inference_model.merge_and_unload()
merged_model.save_pretrained(merged_model_dir)
inference_tokenizer.save_pretrained(merged_model_dir)
print(f"Merged model saved to: {merged_model_dir}")